In [ ]:
%pip --quiet install ollama
%pip --quiet install pydantic

In [38]:
from typing import Optional
from ollama import chat # type: ignore
from pydantic import BaseModel, Field # type: ignore
import json
import tiktoken

## Marker Gene Extraction

In [5]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Permissive class object (allows for missing fields)
class MarkerGene(BaseModel):
    marker_gene_name: Optional[str] = Field(
        None, 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: Optional[str] = Field(
        None, 
        description="The name of the cell type for which this gene serves as a marker."
    )
    tissue_source: Optional[str] = Field(
        None, 
        description="The tissue/location/origin from which this cell type was identified"
    )
    species: Optional[str] = Field(
        None, 
        description="The species from which this marker gene was identified, e.g., 'Homo sapiens', 'Mus musculus'."
    )

class MarkerGeneList(BaseModel):
    genes: List[MarkerGene]  # A list of marker genes extracted from the input text

# Restrictive class object does not allow for missing fields
class MarkerGeneStrict(BaseModel):
    marker_gene_name: str = Field(
        ..., 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: str = Field(
        ..., 
        description="The name of the cell type for which this gene serves as a marker."
    )
    tissue_source: str = Field(
        ..., 
        description="The tissue/location/origin from which this cell type was identified"
    )
    species: str = Field(
        ..., 
        description="The species from which this marker gene was identified, e.g., 'Homo sapiens', 'Mus musculus'."
    )

class MarkerGeneListStrict(BaseModel):
    genes: List[MarkerGeneStrict]  # A list of marker genes extracted from the input text

# class MarkerGene(BaseModel):
#     marker_gene_name: Optional[str] = None
#     cell_name: Optional[str] = None
#     tissue_source: Optional[str] = None

# class MarkerGeneList(BaseModel):
#     genes: list[MarkerGene]

# class MarkerGeneStrict(BaseModel):
#     marker_gene_name: str
#     cell_name: str
#     tissue_source: str

# class MarkerGeneListStrict(BaseModel):
#     genes: list[MarkerGeneStrict]

In [6]:
def extract_genes(user_content, system_prompt, data_model, model="llama3.2"):
    response = chat(
        messages=[
            {
                'role': 'system',
                'content': system_prompt,
            },
            {
                'role': 'user',
                'content': user_content,
            }
        ],
        model = model,
        format = data_model.model_json_schema(),
        options={'temperature': 0},  # Make responses more deterministic

    )
    genes = data_model.model_validate_json(response.message.content)
    return genes.model_dump()["genes"]

In [7]:
system_prompt = """
You are an expert in genomics analyzing scientific literature to extract marker genes for different cell types. 

Your goal is to identify and structure marker gene data from the given text. For each marker gene mentioned, extract:
- The **gene name** (marker_gene_name).
- The **cell type** it is associated with (cell_type_name).
- The **tissue source** where this marker gene was identified (tissue_source).
- The **species** from which the marker gene was studied (species).

If any information is missing or ambiguous, provide the most specific details available. If none of the information is available, the field can be set to null.

The data must be extracted as written in the text, without any modifications.

Return the results in **structured JSON format** with the following schema:
{
    "genes": [
        {
            "marker_gene_name": "GeneX",
            "cell_type_name": "Neuron",
            "tissue_source": "Brain",
            "species": "Homo sapiens"
        },
        ...
    ]
}
"""
user_content = """
After doublet removal and quality filtering, we considered a total of 197,721 cells (106,469 from PG and 91,252 from ING), 
identifying all cell types observed in human WAT (Fig. 2c, d, Supplementary Table 2) with the addition of 
distinct male and female epithelial populations (Dcdc2a+ and Erbb4+, respectively)
"""

system_prompt = "You are a genomics researcher trying to discover different cell types and what genes they express, aka find the marker genes within the given sentence."

In [11]:
models = [
    "llama3.2",
    "deepseek-r1"
]

In [ ]:
genes = extract_genes(user_content, system_prompt, MarkerGeneList, model="llama3.2")
print(len(genes))
print(json.dumps(genes, indent=4))

In [ ]:
genes = extract_genes(user_content, system_prompt, MarkerGeneListStrict, model="llama3.2")
print(len(genes))
print(json.dumps(genes, indent=4))

In [ ]:
genes = extract_genes(user_content, system_prompt, MarkerGeneList, model="deepseek-r1")
print(len(genes))
print(json.dumps(genes, indent=4))

In [ ]:
genes = extract_genes(user_content, system_prompt, MarkerGeneListStrict, model="deepseek-r1")
print(len(genes))
print(json.dumps(genes, indent=4))

# Notes
- Benchmark positive examples and negative examples separately (in each case try out strict and permissive data model)
- Benchmark multiple models
- Group data source data by number of cell type marker gene pairs (i.e. for the same source_rationale, select all evidence derived from it to benchmark)
- Quantify llm extraction capability
    - how well the data was extracted from the input quantify errors for each key/value pair compared to the input text (use the function in analysis/human_vs_human/validate_evidence.ipynb)
    - how much hallucination, i.e. data in extracted but not in input
- Quantify extraction of marker gene to ground truth
    - first ensure the extracted data matches the text verbatim (modify if necessary)
    - second compare the (possibly corrected) extracted data, celltype and gene VVD
    - what other comparisons can we make?

## Comparisons Without Tokenization

In [41]:
def load_human_ev(fn):
    with open(fn, 'r') as f:
        resulting_evidence = json.load(f)
        return [{'extracted':obj['extracted'], 'source':obj['source']} for obj in resulting_evidence] # for the purposes of this comparison, we want to compare what the human has extracted exactly versus the LLM output

In [42]:
load_human_ev("../../data/adipose_Emont2022/evidence_human/evidence.json")

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': ' human ASPC',
   'cell_source': 'adipose',
   'cell_state': 'None',
   'feature_name': 'PDGFRA',
   'feature_type': 'gene'},
  'source': {'source_type': 'text',
   'source_rationale': 'We identified six distinct subpopulations of human ASPCs (see Supplementary Note 1) in subclustered scRNA-seq and sNuc-seq samples, all of which express the common marker gene PDGFRA (Extended Data Fig. 7a, b).',
   'source_id': 'text'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': ' hASPC2',
   'cell_source': 'adipose',
   'cell_state': 'None',
   'feature_name': 'ALDH1A3',
   'feature_type': 'gene'},
  'source': {'source_type': 'text',
   'source_rationale': 'For example, mASPC2 and hASPC2 are characterized by high expression of Aldh1a3 and ALDH1A3, respectively, and strongly resemble previously identified early multipotent progenitor cells that reside in the reticular interstitium of the fat pad5.',
   'source_id'

## Group Human Extracted Data Objects by Source Rationale

In [ ]:
# lalalalala

## Run Comparisons by Groupings of Source Rationales

In [ ]:
# lalalala

## Tokenize LLM Output and User Content

In [13]:
encoding = tiktoken.encoding_for_model("gpt-4o") # TODO: doesnt take in deepseek - can we still do comparable metrics knowing model isn't the same?

In [21]:
def tokenize(input):
    # we want to tokenize the given string and return a list of tokens
    codes = encoding.encode(input)
    return codes

def detokenize(input):
    # takes in an array of tokens and decodes them
    values = encoding.decode(input)
    return values

## Print out Soft Hallucinations Detected by LLM

In [ ]:
genes = extract_genes(user_content, system_prompt, MarkerGeneListStrict, model="llama3.2")

user_content_tokens = tokenize(user_content)
llm_output_gene_tokens = [tokenize(obj['marker_gene_name']) for obj in genes]
llm_output_cell_tokens = [tokenize(obj['cell_type_name']) for obj in genes]

"""
COMMENT: I am not sure if the below code is applicable/relevant. It looks through the tokens of the LLM generated output and checks them against the tokens of the user content.

Tokens may also be split up in different way depending on the sentence structure. In the below example, apparently LLM has "soft hallucinated" 'Er', but this is part of the larger gene Erbb4, which I believe is represented as Errb4+ in the text.
"""
for token_list in llm_output_gene_tokens:
    for tk in token_list:
        if tk not in user_content_tokens:
            print(f"LLM soft hallucinated {encoding.decode_single_token_bytes(tk)} when extracting genes.")

for token_list in llm_output_cell_tokens:
    for tk in token_list:
        if tk not in user_content_tokens:
            print(f"LLM soft hallucinated {encoding.decode_single_token_bytes(tk)} when extracting cells.")

# TODO: compare the output tokens to the user content tokens and we want to ensure that the user content tokens are EXACTLY the same as what the LLM has extracted
# note: we expect this to be false now because the LLM is not returning the tokens exactly as it is, and is "humanizing" it because of how it was trained --> something to be fixed with masking